# Chapter 3 — Giving Agents Tools

In Chapter 1 a tool was a function in a dictionary. That works, and it does not scale:
every tool needs a hand-written schema, nothing is shared between agents, and a tool
added by one team is invisible to every other.

There are three mechanisms, and they differ in who owns the schema, when the schema is
known, and how much reuse you get — not in what the agent can do.




## Setup

This lab installs from **one** `requirements.txt`




In [2]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


repo: /content/aegis


In [3]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.4/695.4 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [4]:
!python tools/check_env.py --chapter 3


dependencies
  ok      langgraph              open-source agent track (StateGraph/END)
  ok      langchain-core         message and tool primitives
  ok      langchain-community    RAGAS dependency — see the pin note
  ok      google-adk             Google Cloud agent track (Agent, Workflow)
  ok      mcp                    tool discovery and hardening (Ch 3, 9, 11)
  ok      openai                 the default real-model tier
  ok      langchain-openai       wires OpenAI into RAGAS
  ok      ragas                  evaluation (Ch 10)
  ok      sacrebleu              required by RAGAS BleuScore
  ok      opentelemetry-sdk      tracing (Ch 10)
  ok      chromadb               vector store (Ch 6)
  ok      rank-bm25              sparse retrieval for hybrid search (Ch 6)
  ok      pytest                 the test suite

critical pin
  ok      langchain-community 0.3.29 (compatible with ragas)

model access
  absent  OPENAI_API_KEY not set
          Offline labs still run: AEGIS_MODEL=mock
  

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [5]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


model tier: mock


Function calling

You hand-write the schema: name, description, and a JSON Schema for the arguments.
The model reads it to decide what to call and with what.

Count the characters. Now imagine forty tools. That cost is the motivation for the
next two mechanisms.


In [6]:
import json, sys
sys.path.insert(0, "labs/chapter-03-tools-function-calling-openapi-mcp")   # this chapter's source lives beside the notebook

from function_calling.tools_fc import (IP_REPUTATION_SCHEMA, SCHEMAS as FC_SCHEMAS,
                                       REGISTRY as FC_REGISTRY, dispatch as fc_dispatch)

print("function calling — hand-written schema:")
print(f'  {len(json.dumps(IP_REPUTATION_SCHEMA))} chars of JSON, written by a human')
print(f'  tools exposed: {[s["function"]["name"] for s in FC_SCHEMAS]}')
print()

verdict = json.loads(fc_dispatch("ip_reputation", {"ip": "203.0.113.42"}))
print("  result:", verdict["verdict"], f'(score {verdict["score"]})')


function calling — hand-written schema:
  255 chars of JSON, written by a human
  tools exposed: ['ip_reputation', 'search_logs']

  result: malicious (score 92)


## Structured output, and validating it before execution

A model emits a tool call as *text*. Text can be wrong in three ways that matter, and
all three arrive looking identical: a tool that does not exist, arguments that do not
match, or output that is not valid JSON at all.

Validation turns each of those into **data** instead of an exception. The rule is
**fail closed**: a rejected call is something you can log, count, and alert on. A crash
is none of those things.

This is also the seam Chapter 11 builds on — the function that rejects a malformed
call is the natural place to reject an *unauthorized* one.


In [7]:
from validation import validate_tool_call, guarded_dispatch

CASES = [
    ("valid call    ", json.dumps({"name": "ip_reputation",
                                   "arguments": {"ip": "203.0.113.42"}})),
    ("unknown tool  ", json.dumps({"name": "delete_all_logs", "arguments": {}})),
    ("bad arguments ", json.dumps({"name": "ip_reputation", "arguments": {"wrong": 1}})),
    ("malformed json", "{not json at all"),
]

for label, raw in CASES:
    outcome = guarded_dispatch(raw, FC_REGISTRY, FC_SCHEMAS)
    result = (outcome["result"][:34] + "...") if outcome["result"] else "-"
    print(f'{label}  ok={str(outcome["ok"]):5} reason={str(outcome["reason"]):15} {result}')

print()
print("Nothing raised. Every rejection is a value the agent can act on.")


valid call      ok=True  reason=accepted        {"ip": "203.0.113.42", "score": 92...
unknown tool    ok=False reason=unknown_tool    -
bad arguments   ok=False reason=bad_arguments   -
malformed json  ok=False reason=malformed_json  -

Nothing raised. Every rejection is a value the agent can act on.


## Generating the tool registry from an OpenAPI spec

Most companies already have an API specification for their internal services. If you
have one, you do not write tool schemas at all — you generate them.

The cell below adds an endpoint to the spec and re-runs the converter. You write no
schema; the agent gains a tool.


In [8]:
import copy
from openapi.tools_openapi import SOC_OPENAPI, openapi_to_schemas

before = openapi_to_schemas(SOC_OPENAPI)
print(f'schemas before: {len(before)}  {[s["function"]["name"] for s in before]}')

extended = copy.deepcopy(SOC_OPENAPI)
extended["paths"]["/identity/user"] = {
    "post": {"operationId": "user_context",
             "summary": "Fetch an account's role, department, and privilege level.",
             "requestBody": {"content": {"application/json": {"schema": {
                 "type": "object",
                 "properties": {"user": {"type": "string"}},
                 "required": ["user"]}}}}}
}

after = openapi_to_schemas(extended)
print(f'schemas after:  {len(after)}  {[s["function"]["name"] for s in after]}')
print()
print("hand-written JSON schema: 0 characters")


schemas before: 2  ['search_logs', 'ip_reputation']
schemas after:  3  ['search_logs', 'ip_reputation', 'user_context']

hand-written JSON schema: 0 characters


## MCP: runtime discovery

The Model Context Protocol inverts the relationship. Instead of the client knowing
what tools exist, a **server advertises** them and any client discovers them at
runtime.

That is a genuinely different property. With function calling and OpenAPI the tool
list is compiled into the client. With MCP, a tool added to the server this morning is
available to every agent this afternoon — no client change, no redeploy.

This uses the real `mcp` SDK over its in-memory transport, so it runs offline.


In [9]:
from mcp.shared.memory import create_connected_server_and_client_session
from mcp_track.tools_mcp import build_soc_server


async def discover_and_call():
    server = build_soc_server()
    async with create_connected_server_and_client_session(server) as client:
        await client.initialize()
        listed = await client.list_tools()                      # DISCOVERY
        names = [t.name for t in listed.tools]
        called = await client.call_tool("ip_reputation", {"ip": "203.0.113.42"})
        return names, json.loads(called.content[0].text)


# Notebook kernels already run an event loop, so asyncio.run() would fail here.
# IPython supports top-level await; scripts (ch03/compare.py) use asyncio.run().
names, result = await discover_and_call()

print("tools discovered at runtime:", names)
print("nothing about these was hard-coded on the client")
print()
print("called ip_reputation ->", result["verdict"])


tools discovered at runtime: ['ip_reputation', 'search_logs', 'user_context']
nothing about these was hard-coded on the client

called ip_reputation -> malicious


## Choosing a strategy

Three mechanisms, one verdict. Check the thing that actually matters, then choose on
the things that differ.


In [10]:
fc_verdict = json.loads(fc_dispatch("ip_reputation", {"ip": "203.0.113.42"}))["verdict"]
from openapi.tools_openapi import dispatch as oa_dispatch
oa_verdict = json.loads(oa_dispatch("ip_reputation", {"ip": "203.0.113.42"}))["verdict"]
mcp_verdict = result["verdict"]

print("verdict by mechanism:")
print("  function calling:", fc_verdict)
print("  openapi:         ", oa_verdict)
print("  mcp:             ", mcp_verdict)
assert fc_verdict == oa_verdict == mcp_verdict
print("  -> identical. the mechanism does not change the answer.")
print()

TRADEOFFS = [
    ("Function calling", "high (hand-write each)", "no", "no"),
    ("OpenAPI", "low (one spec -> N tools)", "via shared spec", "no"),
    ("MCP", "low (server declares)", "yes (any client)", "yes"),
]

print(f'  {"mechanism":18} {"effort per tool":26} {"cross-agent reuse":18} runtime discovery')
for mechanism, effort, reuse, discovery in TRADEOFFS:
    print(f'  {mechanism:18} {effort:26} {reuse:18} {discovery}')
print()
print("Rule of thumb: one tool in one process -> function calling.")
print("An API you already have a spec for -> OpenAPI.")
print("Tools shared across agents or teams -> MCP earns its complexity.")


verdict by mechanism:
  function calling: malicious
  openapi:          malicious
  mcp:              malicious
  -> identical. the mechanism does not change the answer.

  mechanism          effort per tool            cross-agent reuse  runtime discovery
  Function calling   high (hand-write each)     no                 no
  OpenAPI            low (one spec -> N tools)  via shared spec    no
  MCP                low (server declares)      yes (any client)   yes

Rule of thumb: one tool in one process -> function calling.
An API you already have a spec for -> OpenAPI.
Tools shared across agents or teams -> MCP earns its complexity.


### One warning before you ship any of this

Look at what a tool definition actually sends the model: a name, and a **description
in prose**. That description goes into the model's context — and in the MCP case it
comes from a server you may not own.

The callable can be perfectly correct while the prose is the attack. That is tool
poisoning, and Chapter 11 builds the defense: screen the description, fingerprint what
you approved, and detect the rug pull when a server rewrites it later.


---

## What you built

The same tool wired three ways to the same verdict, a validator that fails closed, and
runtime discovery over the real MCP protocol.

- **The mechanism does not change the answer.** It changes schema effort, reuse, and
  whether tools can be discovered rather than compiled in.
- **Validate before you execute.** A rejected call is data; a crash is not.
- **A tool description is an instruction channel.** A registry you do not govern is a
  supply chain you do not control.

**Next:** Chapter 4 gives Aegis a conversation — interviewing an employee about a
suspicious email and producing a structured incident record.
